# DANDI 000004: AMulti-Database Demo

## Overview: Neuronal Electrophysiology Analysis Using [DANDI 000004](https://dandi.github.io/example-notebooks/#dandiset-000004)

Neuroscience is a vast field, subject to domain experts. As such, there are terabytes of publicly available datasets on the topic, running from literature review to experimental design and analysis. One such database containing these datasets is DANDI: Distributed Archives for Neurophysiology Data Integration. Considered the gold standard for electrophysiology, DANDI uses the .nwb format (Neurodata Without Borders, a proprietary data format, organized into “Dandisets”) that plays nicely with vector databases, graph databases, and tabular databases. We explain our proposed approach below, using Dandiset 000004, “A NWB-based dataset and processing pipeline of human single-neuron activity during a declarative memory task”.

More on the DANDI API: https://dandi.readthedocs.io/en/latest/modref/dandiapi.html

This notebook demonstrates querying data that was ingested from a single streamed NWB session
(DANDI dandiset 000004) into the following databases:

| Database | Type | Purpose |
|---|---|---|
| **Neo4j** | Graph | Brain hierarchy (Subject -> Session -> Neuron -> Region) |
| **PostgreSQL** | Tabular | Subjects, sessions, neurons, trials |

In [1]:
import os
from dotenv import load_dotenv

# sanity checks
load_dotenv()
# print(os.environ.get("PG_DSN"))
# print(os.environ.get("NEO4J_URI"))

True

In [2]:
# create tables from schema.sql
import psycopg, os
from dotenv import load_dotenv
load_dotenv()

schema_path = os.path.join(os.path.dirname(os.path.abspath('.')), 'experiments', 'database', 'schema.sql')
schema_path = 'schema.sql'

with open(schema_path) as f:
    sql = f.read()

with psycopg.connect(os.environ['PG_DSN']) as conn, conn.cursor() as cur:
    cur.execute(sql)
    conn.commit()

In [3]:
# get data from DANDI
# run this before anything else
%run -i "data/ingest.py"

Session: H11_9
[Postgres] 25 neurons, 200 trials.
[Neo4j] 25 neurons.


In [6]:
import sys
import os
import pandas as pd
from pynwb import NWBHDF5IO

# make sure to change this path to the correct NWB file
# usually when you change NWB_URL and NWB_PATH in ingest.py,
# the NWB file will be downloaded to the same path
nwb_file_path = "data/sub-P11HMH_ses-20061101_ecephys+image.nwb"

# Open the NWB file in read mode
io = NWBHDF5IO(nwb_file_path, 'r')
nwbfile = io.read()
display(nwbfile)

Data type,object
Shape,"(1004,)"
Array size,7.84 KiB
Chunk shape,None
Compression,None
Compression opts,None
Uncompressed size (bytes),8032
Compressed size (bytes),16064
Compression ratio,0.5
Data type,float64
Shape,"(1004,)"


## Neo4j: Graph Queries

Node breakdown:

```
(Subject)-[:HAS_SESSION]->(Session)-[:HAS_NEURON]->(Neuron)-[:LOCATED_IN]->(BrainRegion)
(Session)-[:PRESENTED]->(Stimulus)
```

In [ ]:
from utils.neo4j import *

In [ ]:
# graph node counts
summary = get_graph_summary()
print('Graph node counts:')
print(summary)

## PostgreSQL: Neuron Firing Statistics

After getting the data from Neo4j, we can refine our queries (and questions) even more with additional Postgres transactions.

In [ ]:
from utils.postgres import *

In [ ]:
# session overview
sessions = get_session_summary()
print(f'Ingested sessions: {len(sessions)}')
print(sessions)

In [ ]:
# firing statistics for ALL regions ALL sessions
stats = firing_stats()
print('Firing statistics by brain region (all sessions):')
print(stats)

In [ ]:
# filter: firing stats only for HIPPOCAMPAL neurons
hipp_stats = firing_stats_hippocampus()
print('Firing statistics for Hippocampus neurons (all sessions):')
print(stats)